# Qwen3.5-4B SFT Training with Unsloth (10k Samples)

This notebook contains the complete interactive pipeline to fine-tune the Qwen 3.5 4B model using Unsloth on the `upwitu/trash_draft_am` dataset.

In [ ]:
import os
import sys

# THIẾT LẬP THIẾT BỊ CUDA VÀ PHÂN BỔ BỘ NHỚ TRƯỚC KHI IMPORT TORCH
# Chỉ định sử dụng GPU 1
os.environ["CUDA_VISIBLE_DEVICES"] = "1"
# Kích hoạt expandable_segments để tránh phân mảnh bộ nhớ CUDA
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch
import torch.utils

print(f"CUDA_VISIBLE_DEVICES set to: {os.environ.get('CUDA_VISIBLE_DEVICES')}")
if torch.cuda.is_available():
    print(f"Current CUDA Device: {torch.cuda.current_device()} - {torch.cuda.get_device_name(0)}")
else:
    print("CUDA is not available")

# 1. SYSTEM COMPATIBILITY PATCHES (Monkey patching older PyTorch versions)
print("Applying system compatibility patches...")
for i in range(1, 8):
    int_attr = f"int{i}"
    uint_attr = f"uint{i}"
    if not hasattr(torch, int_attr):
        setattr(torch, int_attr, torch.int8)
    if not hasattr(torch, uint_attr):
        setattr(torch, uint_attr, torch.uint8)

if not hasattr(torch.utils, "_pytree"):
    import torch.utils._pytree
if not hasattr(torch.utils._pytree, "register_constant"):
    torch.utils._pytree.register_constant = lambda cls: cls

# 2. Import unsloth FIRST, before transformers/trl/peft.
# Unsloth must be imported before any other HuggingFace libraries to apply all optimizations.
from unsloth import FastLanguageModel, train_on_responses_only

# 3. Patch transformers AFTER unsloth has done its own init
import transformers
transformers.utils.import_utils.is_torchao_available = lambda *args, **kwargs: False
transformers.utils.is_torchao_available = lambda *args, **kwargs: False
if hasattr(transformers.utils.import_utils, "_torchao_available"):
    transformers.utils.import_utils._torchao_available = False

# Unset DDP environment variables for safe single GPU training
for key in ["WORLD_SIZE", "RANK", "LOCAL_RANK", "MASTER_ADDR", "MASTER_PORT"]:
    if key in os.environ:
        del os.environ[key]

print("System compatibility patches applied successfully!")

In [ ]:
import gc
def clear_gpu_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        print("Released GPU cache memory.")

clear_gpu_memory()

In [ ]:
# Load remaining libraries (unsloth already imported in cell 1)
from datasets import load_dataset, Dataset
from trl import SFTTrainer, SFTConfig

# DataCollatorForCompletionOnlyLM location changed or was deprecated in modern TRL.
# We define a lightweight fallback using DataCollatorForLanguageModeling if TRL's class is missing.
DataCollatorForCompletionOnlyLM = None
for _module in ("trl", "trl.trainer", "trl.trainer.utils", "trl.data_utils"):
    try:
        import importlib
        _mod = importlib.import_module(_module)
        DataCollatorForCompletionOnlyLM = getattr(_mod, "DataCollatorForCompletionOnlyLM", None)
        if DataCollatorForCompletionOnlyLM is not None:
            print(f"DataCollatorForCompletionOnlyLM imported from: {_module}")
            break
    except ImportError:
        continue

if DataCollatorForCompletionOnlyLM is None:
    print("DataCollatorForCompletionOnlyLM not found in TRL. Loading custom fallback...")
    from transformers import DataCollatorForLanguageModeling
    class FallbackCompletionOnlyCollator(DataCollatorForLanguageModeling):
        def __init__(self, response_template, instruction_template, tokenizer, *args, **kwargs):
            super().__init__(tokenizer=tokenizer, mlm=False, *args, **kwargs)
            self.response_token_ids = response_template
            self.instruction_token_ids = instruction_template
        def torch_call(self, examples):
            batch = super().torch_call(examples)
            labels = batch["labels"].clone()
            for i in range(labels.shape[0]):
                seq = batch["input_ids"][i].tolist()
                resp_idx = -1
                for idx in range(len(seq) - len(self.response_token_ids) + 1):
                    if seq[idx:idx+len(self.response_token_ids)] == self.response_token_ids:
                        resp_idx = idx + len(self.response_token_ids)
                        break
                if resp_idx != -1:
                    labels[i, :resp_idx] = -100
            batch["labels"] = labels
            return batch
    DataCollatorForCompletionOnlyLM = FallbackCompletionOnlyCollator
    print("Fallback collator ready.")

MODEL_NAME = "Qwen/Qwen3.5-4B"
PERSISTENT_DIR = "./outputs"
ADAPTER_PATH = os.path.join(PERSISTENT_DIR, "sft_lora_adapter")
MERGED_PATH = os.path.join(PERSISTENT_DIR, "sft_merged_model")
os.makedirs(PERSISTENT_DIR, exist_ok=True)
print("Libraries loaded successfully.")

## Load and Split Dataset (10,000 samples)

In [ ]:
import pandas as pd
import json

print("Loading dataset upwitu/trash_draft_am...")
try:
    dataset = load_dataset("upwitu/trash_draft_am", split="train")
    print("Standard load_dataset succeeded.")
except Exception as e:
    print(f"Standard load_dataset failed: {e}")
    print("Falling back to direct parquet read via pandas...")
    dataset_url = "hf://datasets/upwitu/trash_draft_am@refs/convert/parquet/default/train/0000.parquet"
    df = pd.read_parquet(dataset_url)
    dataset = Dataset.from_pandas(df, preserve_index=False)
    print(f"Parquet fallback loaded. Columns: {dataset.column_names}")

print(f"Original dataset size: {len(dataset)}")

# --- Inspect first row to understand types ---
row0 = dataset[0]
print("\nColumn types for row[0]:")
for k, v in row0.items():
    print(f"  {k}: type={type(v).__name__}, len={len(v) if hasattr(v, '__len__') else 'n/a'}, sample={repr(str(v)[:120])}")

# Select 10,000 samples
print("\nSelecting 10,000 samples for SFT...")
dataset_subset = dataset.select(range(min(10000, len(dataset))))

dataset_split = dataset_subset.train_test_split(test_size=0.1, seed=42)
train_dataset = dataset_split["train"]
eval_dataset  = dataset_split["test"]
print(f"Dataset ready: {len(train_dataset)} train, {len(eval_dataset)} evaluation samples")

## Initialize Model and LoRA (Unsloth)

In [ ]:
print("Loading model via Unsloth...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=2048,
    dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    load_in_4bit=True,
    trust_remote_code=True
)

# Unsloth may return a VL Processor (e.g. Qwen3VLProcessor) that wraps the real
# text tokenizer inside a .tokenizer attribute. The Processor supports
# apply_chat_template() but NOT .encode() / .decode().  Extract the underlying
# tokenizer so we can call .encode() where needed.
text_tokenizer = getattr(tokenizer, "tokenizer", tokenizer)
print(f"tokenizer type  : {type(tokenizer).__name__}")
print(f"text_tokenizer  : {type(text_tokenizer).__name__}")

print("Configuring LoRA PEFT...")
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,
    lora_dropout=0.0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)
print("LoRA configured successfully.")
model.print_trainable_parameters()

## Diagnose Format Function on First Row

In [ ]:
# Dry-run format_sft_data on first 3 rows to catch errors before bulk map
def _parse_messages(raw):
    """Safely parse augmented_messages regardless of input type."""
    if raw is None:
        return None
    if isinstance(raw, str):
        return json.loads(raw)
    if isinstance(raw, (list, tuple)):
        # Each element might itself be a dict or a JSON string
        result = []
        for item in raw:
            if isinstance(item, dict):
                result.append(item)
            elif isinstance(item, str):
                result.append(json.loads(item))
            else:
                result.append(dict(item))
        return result
    # Try dict-like
    try:
        return [dict(m) for m in raw]
    except Exception:
        return None

print("Diagnosing format_sft_data on first 3 rows...")
for idx in range(min(3, len(train_dataset))):
    row = train_dataset[idx]
    raw = row.get("augmented_messages")
    print(f"\n--- Row {idx} ---")
    print(f"  augmented_messages type: {type(raw).__name__}")
    try:
        messages = _parse_messages(raw)
        if messages is None or len(messages) == 0:
            print("  SKIP: messages is None or empty")
            continue
        print(f"  Parsed {len(messages)} messages. Roles: {[m.get('role', '?') for m in messages]}")
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        print(f"  Template OK. Output length: {len(text)} chars")
        print(f"  Preview: {repr(text[:200])}")
    except Exception as e:
        import traceback
        print(f"  ERROR: {e}")
        traceback.print_exc()

## Formatting Data

In [ ]:
_format_errors = []
_MAX_ERROR_LOG = 3

def format_sft_data(row):
    """
    Format dataset rows using apply_chat_template.
    Returns {'text': None} for rows that fail so they can be filtered out.
    Logs the first _MAX_ERROR_LOG exceptions for diagnostics.
    """
    global _format_errors
    try:
        raw = row.get("augmented_messages")
        messages = _parse_messages(raw)
        if messages is None or len(messages) == 0:
            return {"text": None}

        # Do NOT pass tools= to apply_chat_template.
        # The system prompt already embeds tool definitions as plain text,
        # and passing tools= triggers strict alternating-role validation that fails
        # for datasets with tool/assistant multi-turn patterns.
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False,
        )
        return {"text": text}
    except Exception as exc:
        if len(_format_errors) < _MAX_ERROR_LOG:
            _format_errors.append(str(exc))
        return {"text": None}

print("Mapping datasets to templates...")
_format_errors = []
train_dataset = train_dataset.map(format_sft_data, remove_columns=train_dataset.column_names)
eval_dataset  = eval_dataset.map(format_sft_data,  remove_columns=eval_dataset.column_names)

if _format_errors:
    print(f"\nFirst {len(_format_errors)} formatting errors encountered:")
    for i, err in enumerate(_format_errors):
        print(f"  [{i}] {err}")

# Filter out rows that failed formatting
train_dataset = train_dataset.filter(lambda x: x["text"] is not None)
eval_dataset  = eval_dataset.filter(lambda x: x["text"] is not None)

print(f"After filtering: {len(train_dataset)} train, {len(eval_dataset)} eval samples")

assert len(train_dataset) > 0, (
    "FATAL: train_dataset is empty after formatting! "
    f"Errors: {_format_errors}"
)
print("Preview:")
print(train_dataset[0]["text"][:600] + "\n...\n")

## QA: Validate Dataset and Confirm Response Pattern Tokens

In [ ]:
assert len(train_dataset) > 0, "ERROR: train_dataset is empty after formatting!"
assert len(eval_dataset)  > 0, "ERROR: eval_dataset is empty after formatting!"

sample_texts = [train_dataset[i]["text"] for i in range(min(20, len(train_dataset)))]
has_think = sum(1 for t in sample_texts if "<think>" in t)
has_asst  = sum(1 for t in sample_texts if "<|im_start|>assistant" in t)

print(f"Dataset sizes - train: {len(train_dataset)}, eval: {len(eval_dataset)}")
print(f"Sampled 20 rows - '<think>': {has_think}/20 | assistant turn: {has_asst}/20")

if has_think > 0:
    RESPONSE_PART = "<|im_start|>assistant\n<think>\n"
    print("Thinking tags found -> RESPONSE_PART includes <think>")
else:
    RESPONSE_PART = "<|im_start|>assistant\n"
    print("No <think> tags -> plain RESPONSE_PART")

INSTRUCTION_PART = "<|im_start|>user\n"

# text_tokenizer is the underlying tokenizer extracted from the processor in the model cell.
# Use it for .encode() / .decode() calls; the outer processor does not support these methods.
resp_ids   = text_tokenizer.encode(RESPONSE_PART,    add_special_tokens=False)
instr_ids  = text_tokenizer.encode(INSTRUCTION_PART, add_special_tokens=False)
sample_ids = text_tokenizer.encode(sample_texts[0],  add_special_tokens=False)

def ids_in_seq(needle, haystack):
    n = len(needle)
    return any(haystack[i:i+n] == needle for i in range(len(haystack) - n + 1))

resp_found  = ids_in_seq(resp_ids,  sample_ids)
instr_found = ids_in_seq(instr_ids, sample_ids)

print(f"\nToken-level check on sample[0]:")
print(f"  INSTRUCTION_PART {instr_ids} found: {instr_found}")
print(f"  RESPONSE_PART    {resp_ids}  found: {resp_found}")

if not resp_found:
    print("\nWARNING: RESPONSE_PART tokens NOT found in tokenized output!")
    print("   train_on_responses_only will produce 0 matching samples.")
    print("   Sample text (first 400 chars):\n", sample_texts[0][:400])
else:
    print("\nRESPONSE_PART tokens confirmed present in sample.")

print(f"\nINSTRUCTION_PART : {repr(INSTRUCTION_PART)}")
print(f"RESPONSE_PART    : {repr(RESPONSE_PART)}")

## Training Configuration

In [ ]:
# Tăng tốc huấn luyện bằng cách tăng per_device_train_batch_size lên 4
# và giảm gradient_accumulation_steps xuống 4 để giảm số bước lặp phụ không cần thiết
sft_config = SFTConfig(
    dataset_text_field="text",
    max_seq_length=2048,
    packing=False,
    output_dir=PERSISTENT_DIR,
    learning_rate=2e-5,
    per_device_train_batch_size=4, # Tăng lên 4 để xử lý song song nhanh hơn
    gradient_accumulation_steps=4, # Giảm xuống 4 duy trì Global Batch Size = 4 * 4 = 16
    num_train_epochs=3.0,
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    fp16=False,
    bf16=True,
    optim="paged_adamw_8bit",
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    logging_steps=5,
    report_to="none"
)

# Pass text_tokenizer (not the outer processor) to SFTTrainer.
# SFTTrainer needs a true tokenizer with .encode() / .pad() for data collation.
trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    tokenizer=text_tokenizer
)

trainer = train_on_responses_only(
    trainer,
    instruction_part=INSTRUCTION_PART,
    response_part=RESPONSE_PART,
)

n_train = len(trainer.train_dataset)
n_eval  = len(trainer.eval_dataset)
print(f"After train_on_responses_only: {n_train} train, {n_eval} eval samples")

if n_train == 0:
    print("WARNING: train_on_responses_only produced 0 samples. Falling back to DataCollatorForCompletionOnlyLM...")
    resp_ids_list  = text_tokenizer.encode(RESPONSE_PART,    add_special_tokens=False)
    instr_ids_list = text_tokenizer.encode(INSTRUCTION_PART, add_special_tokens=False)
    collator = DataCollatorForCompletionOnlyLM(
        response_template=resp_ids_list,
        instruction_template=instr_ids_list,
        tokenizer=text_tokenizer,
    )
    trainer = SFTTrainer(
        model=model,
        args=sft_config,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        tokenizer=text_tokenizer,
        data_collator=collator,
    )
    n_train = len(trainer.train_dataset)
    n_eval  = len(trainer.eval_dataset)
    print(f"Fallback trainer: {n_train} train, {n_eval} eval samples")

assert n_train > 0, (
    "FATAL: train_dataset has 0 samples even after fallback!\n"
    "Check QA cell token-level output. RESPONSE_PART tokens may not be in the data."
)
print(f"Trainer ready: {n_train} train, {n_eval} eval samples.")

## Run Training

In [ ]:
import logging
import sys
import time

print("Configuring log file output...")
log_filepath = os.path.join(PERSISTENT_DIR, "training_output.log")

# Setup logging to direct outputs both to the notebook console and a file
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    handlers=[
        logging.FileHandler(log_filepath, mode='w', encoding='utf-8'),
        logging.StreamHandler(sys.stdout)
    ]
)

# Custom stdout redirector to capture standard print statements from the training loop
class DualOutput:
    def __init__(self, stream, filepath):
        self.stream = stream
        self.file = open(filepath, 'a', encoding='utf-8')
    def write(self, data):
        self.stream.write(data)
        self.file.write(data)
        self.file.flush()
    def flush(self):
        self.stream.flush()
        self.file.flush()
    def close(self):
        self.file.close()

print(f"Redirecting stdout/stderr. All training logs will be saved to: {log_filepath}")
original_stdout = sys.stdout
original_stderr = sys.stderr
dual_stdout = DualOutput(original_stdout, log_filepath)
dual_stderr = DualOutput(original_stderr, log_filepath)

sys.stdout = dual_stdout
sys.stderr = dual_stderr

try:
    print("="*50)
    print(f"STARTING SFT TRAINING LOOP AT {time.strftime('%Y-%m-%d %H:%M:%S')}")
    print("="*50)
    print("--- MODEL & DATA INFO ---")
    print(f"Model Name                  : {MODEL_NAME}")
    print(f"Dataset Size (Train)        : {n_train}")
    print(f"Dataset Size (Eval)         : {n_eval}")
    print(f"Output Directory            : {PERSISTENT_DIR}")
    print(f"Adapter Path                : {ADAPTER_PATH}")
    print(f"Merged Model Path           : {MERGED_PATH}")
    print("\n--- TRAINING HYPERPARAMETERS ---")
    print(f"Learning Rate               : {sft_config.learning_rate}")
    print(f"Per-Device Train Batch Size : {sft_config.per_device_train_batch_size}")
    print(f"Gradient Accumulation Steps: {sft_config.gradient_accumulation_steps}")
    print(f"Global Batch Size           : {sft_config.per_device_train_batch_size * sft_config.gradient_accumulation_steps}")
    print(f"Epochs                      : {sft_config.num_train_epochs}")
    print(f"Optimizer                   : {sft_config.optim}")
    print(f"Precision                   : {'bfloat16' if sft_config.bf16 else 'float16'}")
    print(f"Max Sequence Length         : {sft_config.max_seq_length}")
    print(f"Packing Enabled             : {sft_config.packing}")
    print(f"Active GPU (Device ID)      : {torch.cuda.current_device()} ({torch.cuda.get_device_name(0)})")
    print(f"Total GPU VRAM Allocated    : {torch.cuda.memory_allocated() / (1024**3):.2f} GB")
    print("="*50 + "\n")
    
    trainer.train()
    print("\n" + "="*50)
    print(f"TRAINING FINISHED SUCCESSFULLY AT {time.strftime('%Y-%m-%d %H:%M:%S')}")
    print("="*50)
except Exception as e:
    print("Failure during SFT training loop:", str(e))
    raise e
finally:
    # Always restore original streams
    sys.stdout = original_stdout
    sys.stderr = original_stderr
    dual_stdout.close()
    dual_stderr.close()
    print("Stdout and stderr restored. Log file closed.")

## Plot Training Loss

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

if hasattr(trainer, "state") and trainer.state.log_history:
    history = trainer.state.log_history
    train_steps, train_losses = [], []
    eval_steps,  eval_losses  = [], []
    for log in history:
        step = log.get("step")
        if "loss"      in log: train_steps.append(step); train_losses.append(log["loss"])
        if "eval_loss" in log: eval_steps.append(step);  eval_losses.append(log["eval_loss"])
    plt.figure(figsize=(10, 5))
    if train_losses: plt.plot(train_steps, train_losses, label="Train Loss",      color="red",  marker="o")
    if eval_losses:  plt.plot(eval_steps,  eval_losses,  label="Validation Loss", color="blue", marker="s")
    plt.xlabel("Step"); plt.ylabel("Loss")
    plt.title("SFT Loss Curve")
    plt.legend(); plt.grid(True); plt.show()
else:
    print("No log history found.")

## Save PEFT Adapter and Merge Model

In [ ]:
print(f"Saving adapter to: {ADAPTER_PATH}")
model.save_pretrained(ADAPTER_PATH)
tokenizer.save_pretrained(ADAPTER_PATH)

print("Merging and saving model weights (16-bit)...")
model.save_pretrained_merged(MERGED_PATH, tokenizer, save_method="merged_16bit")
print(f"Merged model saved successfully to: {MERGED_PATH}")